# 🌴 Guanacaste Tourism Forecaster — Fase 3: Data Ingestion & ETL
### Consolidación de Inteligencia Turística y Macroeconómica (2009-2026)
---
**Estado:** Preparación de Datos ✅ | **Siguiente Fase:** Modelado con Prophet 🔮

In [ ]:
import pandas as pd
import numpy as np
import os
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import yfinance as yf
from itables import show as itable
import itables.options as opt

TEMPLATE = 'plotly_dark'
PLOTLY_LOADED = False

# Opciones globales de itables
opt.lengthMenu = [10, 25, 50, 100]
opt.pageLength = 15
opt.style = 'table-layout:auto;width:100%;'

def show_plot(fig, height=500):
    global PLOTLY_LOADED
    include_js = not PLOTLY_LOADED
    if not PLOTLY_LOADED:
        PLOTLY_LOADED = True
    html = fig.to_html(full_html=False, include_plotlyjs=include_js)
    display(HTML(f'<div style="height:{height}px;width:100%">{html}</div>'))

print('✅ Listo. Plotly + itables configurados.')

### 1. Ingesta Macroeconómica — FRED + Yahoo Finance

In [ ]:
start_date = '2009-01-01'

series = {'UNRATE': 'tasa_desempleo_usa', 'CPIAUCSL': 'inflacion_usa_cpi'}
df_list = []
for s_id, s_name in series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={s_id}'
    df_t = pd.read_csv(url, parse_dates=['observation_date'], na_values='.')
    df_t = df_t[df_t['observation_date'] >= start_date].set_index('observation_date')
    df_t = df_t.resample('ME').mean().rename(columns={s_id: s_name})
    df_list.append(df_t)

df_macro = pd.concat(df_list, axis=1).reset_index().rename(columns={'observation_date': 'DATE'})

print('⬇️ Descargando WTI desde Yahoo Finance...')
oil = yf.download('CL=F', start=start_date, interval='1mo', progress=False)
if isinstance(oil.columns, pd.MultiIndex):
    oil_prices = oil['Close'].iloc[:, 0]
else:
    oil_prices = oil['Close']
oil_df = oil_prices.resample('ME').mean().rename('precio_petroleo_wti').reset_index()
oil_df.rename(columns={'Date': 'DATE'}, inplace=True)
df_macro = pd.merge(df_macro, oil_df, on='DATE', how='left')

print(f'✅ Macro lista: {df_macro.shape[0]} meses.')

### 2. Data Infill — Parche 2025-2026 (BLS / EIA)

In [ ]:
PATCH_DATA = {
    '2026-01-31': {'tasa_desempleo_usa': 4.3, 'inflacion_usa_cpi': 250.5, 'precio_petroleo_wti': 60.04},
    '2026-02-28': {'tasa_desempleo_usa': 4.4, 'inflacion_usa_cpi': 251.2, 'precio_petroleo_wti': 64.51},
    '2026-03-31': {'tasa_desempleo_usa': 4.3, 'inflacion_usa_cpi': 252.8, 'precio_petroleo_wti': 91.38}
}

for dt_str, values in PATCH_DATA.items():
    ts = pd.to_datetime(dt_str)
    mask = df_macro['DATE'] == ts
    if mask.any():
        for col, val in values.items():
            df_macro.loc[mask & df_macro[col].isna(), col] = val

print('✅ Infill completado.')
print(df_macro[['tasa_desempleo_usa','inflacion_usa_cpi','precio_petroleo_wti']].isna().sum())

### 3. Fusión Maestra — Merge Final

In [ ]:
df_arr = pd.read_csv('../data/processed/arrivals_clean.csv')
df_arr['DATE'] = pd.to_datetime(df_arr[['Year', 'Month']].assign(day=1)) + pd.offsets.MonthEnd(0)

df_occ = pd.read_csv('../data/processed/occupancy_clean.csv')
df_occ['DATE'] = pd.to_datetime(df_occ[['Year', 'Month']].assign(day=1)) + pd.offsets.MonthEnd(0)

df_master = pd.merge(df_arr, df_occ[['DATE', 'Guanacaste_Occupancy_Pct']], on='DATE', how='left')
df_master = pd.merge(df_master, df_macro, on='DATE', how='left')

os.makedirs('../data/merged', exist_ok=True)
df_master.to_csv('../data/merged/master_tourism_data.csv', index=False)

print(f'✅ Master dataset: {df_master.shape[0]} filas × {df_master.shape[1]} columnas')

---
### 📋 Explorador de Datos Interactivo
Puedes buscar, ordenar, filtrar y paginar. Haz clic en cualquier columna para ordenar.

In [ ]:
# Seleccionamos las columnas clave para la vista
cols_vista = [
    'DATE',
    'Arrivals_sjo',
    'Arrivals_lir',
    'Guanacaste_Occupancy_Pct',
    'tasa_desempleo_usa',
    'inflacion_usa_cpi',
    'precio_petroleo_wti'
]

df_vista = df_master[cols_vista].copy()

# Renombrar para que se vea más claro
df_vista.columns = [
    'Fecha',
    'Llegadas SJO',
    'Llegadas LIR',
    'Ocupación Guana (%)',
    'Desempleo USA (%)',
    'Inflación CPI',
    'Petróleo WTI (USD)'
]

# Formato de fechas legible
df_vista['Fecha'] = df_vista['Fecha'].dt.strftime('%b %Y')

# Redondear decimales
df_vista = df_vista.round(2)

print(f'📊 Mostrando {len(df_vista)} meses de datos (2009 → 2026)')
itable(
    df_vista,
    caption='Dataset Maestro — Guanacaste Tourism Intelligence',
    scrollY='400px',
    scrollCollapse=True,
    paging=True,
    searching=True
)

---
## 📊 Análisis Visual Interactivo

In [ ]:
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'Llegadas Internacionales — SJO vs LIR (2009-2026)',
        'Ocupación Hotelera en Guanacaste (%)'
    ),
    vertical_spacing=0.14
)

fig.add_trace(go.Scatter(
    x=df_master['DATE'], y=df_master['Arrivals_sjo'],
    name='SJO — Alajuela', line=dict(color='#00d4ff', width=2, dash='dot'),
    fill='tozeroy', fillcolor='rgba(0,212,255,0.08)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_master['DATE'], y=df_master['Arrivals_lir'],
    name='LIR — Guanacaste', line=dict(color='#00ff9d', width=2),
    fill='tozeroy', fillcolor='rgba(0,255,157,0.08)'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_master['DATE'], y=df_master['Guanacaste_Occupancy_Pct'],
    name='Ocupación (%)', line=dict(color='#ff6b35', width=3),
    fill='tozeroy', fillcolor='rgba(255,107,53,0.1)'
), row=2, col=1)

fig.add_vrect(
    x0='2020-03-01', x1='2021-12-31',
    fillcolor='rgba(255,0,0,0.08)', line_width=0,
    annotation_text='COVID-19', annotation_position='top left',
    annotation_font_color='#ff4b4b'
)

fig.update_layout(
    title='<b>Dinamismo Turístico de Guanacaste (2009-2026)</b>',
    template=TEMPLATE, height=600, hovermode='x unified',
    legend=dict(orientation='h', y=-0.12),
    font=dict(family='Arial, sans-serif')
)
fig.update_yaxes(title_text='Llegadas / mes', row=1, col=1)
fig.update_yaxes(title_text='Ocupación %', row=2, col=1)

show_plot(fig, height=620)

In [ ]:
fig_macro = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Desempleo USA (%)', 'Inflación USA (CPI)', 'Petróleo WTI (USD/bbl)')
)

macro_cols = [
    ('tasa_desempleo_usa',  '#f72585'),
    ('inflacion_usa_cpi',   '#4cc9f0'),
    ('precio_petroleo_wti', '#f9c74f')
]

for i, (col, color) in enumerate(macro_cols, 1):
    datos = df_master[['DATE', col]].dropna()
    fig_macro.add_trace(go.Scatter(
        x=datos['DATE'], y=datos[col],
        name=col, line=dict(color=color, width=2), showlegend=False
    ), row=1, col=i)

fig_macro.update_layout(
    title='<b>Indicadores Macroeconómicos USA — Drivers del Turismo</b>',
    template=TEMPLATE, height=360,
    font=dict(family='Arial, sans-serif')
)
show_plot(fig_macro, height=380)

In [ ]:
corr_cols   = ['Arrivals_sjo', 'Arrivals_lir', 'Guanacaste_Occupancy_Pct', 'tasa_desempleo_usa', 'precio_petroleo_wti']
corr_labels = ['SJO', 'LIR', 'Ocupación Guana', 'Desempleo USA', 'Petróleo WTI']

corr_matrix = df_master[corr_cols].dropna().corr()

fig_corr = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdYlGn',
    x=corr_labels, y=corr_labels,
    template=TEMPLATE,
    title='<b>Matriz de Correlación — Drivers de Ocupación Hotelera</b>'
)
fig_corr.update_layout(height=420, font=dict(family='Arial, sans-serif'))
show_plot(fig_corr, height=440)